In [32]:
import h5py
import anndata
import pandas as pd
import scipy.sparse as scs
import hisepy as hp
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [3]:
df = pd.read_csv('scrna_metadata.csv')
pbmcs = df[df['tissue'] == 'PBMC']

mm_obs = pd.read_parquet('DataScales/build_megazarr/MM_obs.parquet')

In [27]:
# Cell 3 — h5 reader helpers
def read_mat(h5_con, mat_name):
  return scs.csc_matrix(
      (h5_con[mat_name]["data"][:], h5_con[mat_name]["indices"][:], h5_con[mat_name]["indptr"][:]),
      shape=tuple(h5_con[mat_name]["shape"][:]),
  )

def read_feats(h5_con, mat_name, name_col):
  feats = h5_con[mat_name]["features"][name_col][:]
  return [x.decode("UTF-8") for x in feats]

def read_obs(h5con):
  bc = [x.decode("UTF-8") for x in h5con["matrix"]["barcodes"][:]]
  obs_df = pd.DataFrame({"barcodes": bc})
  for col in h5con["matrix"]["observations"].keys():
      values = h5con["matrix"]["observations"][col][:]
      if isinstance(values[0], (bytes, bytearray)):
          values = [x.decode("UTF-8") for x in values]
      obs_df[col] = values
  return obs_df

# Cell 4 — build adata (no metadata)
def build_adata(h5_file):
  h5_con = h5py.File(h5_file, mode="r")
  rna_mat = read_mat(h5_con, "matrix")
  obs = read_obs(h5_con)
  genes = read_feats(h5_con, "matrix", "name")
  h5_con.close()
  adata = anndata.AnnData(X=rna_mat.T, obs=obs.set_index("barcodes"))
  adata.var_names = genes
  adata.var_names_make_unique()
  return adata

In [28]:
mm_uuids = list(pbmcs_mm['file.id'])
mm_pull = hp.reader.cache_files(mm_uuids)

2026-07-30 15:30:34,108 INFO [hisepy.logging:185] logging 17527 134849873725248 Calling cache_files
2026-07-30 15:36:15,416 INFO [hisepy.logging:228] logging 17527 134849873725248 Finished cache_files successfully (time_elapsed=338.548s)


In [37]:
mm_obs = pd.read_parquet('DataScales/build_megazarr/MM_obs.parquet')

kit_ids = mm_obs['sample.sampleKitGuid'].unique()
pbmcs_mm = pbmcs[pbmcs['sample.sampleKitGuid'].isin(kit_ids)]

In [40]:
adata_list = []
with ThreadPoolExecutor(max_workers=16) as executor:
  futures = {executor.submit(build_adata, p): p for p in mm_pull}
  for future in tqdm(as_completed(futures), total=len(mm_pull)):
      adata_list.append(future.result())

adata = anndata.concat(adata_list)
adata = adata[adata.obs_names.isin(mm_obs["barcodes"])]

100%|██████████| 157/157 [02:05<00:00,  1.25it/s]


In [44]:
import subprocess 

adata.write_h5ad("mm_raw_33538.h5ad")
subprocess.run(["gsutil", "cp", "mm_raw_33538.h5ad", "gs://imm-zarr-poc/MegaZarr/mm_raw_33538.h5ad"], check=True)

/home/workspace/environment/minimalv4/lib/python3.11/site-packages/anndata/_core/anndata.py:1255: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'batch_id' as categorical
/home/workspace/environment/minimalv4/lib/python3.11/site-packages/anndata/_core/anndata.py:1255: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'cell_name' as categorical
/home/workspace/environment/minimalv4/lib/python3.11/site-packages/anndata/_core/anndata.py:1255: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'chip_id' as categorical
/home/workspace/environment/minimalv4/lib/python3.11/site-packages/anndata/_core/anndata.py:1255: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'hto_barcode' as categ

CompletedProcess(args=['gsutil', 'cp', 'mm_raw_33538.h5ad', 'gs://imm-zarr-poc/MegaZarr/mm_raw_33538.h5ad'], returncode=0)